In [ ]:
import json
from pathlib import Path


In [ ]:
basepath = "/home/hmack/Development/rodent_experiments/runs"

detect_paths = ["detect001", "detect005", "detect01", "detect02"]

model_paths = [
    "yoloe_rodentornot",
    "yoloe_animalornot",
    "yolo26",
    "speciesnet",
    "biotrove-clip_species_names",
    "biotrove-clip_rodent_or_not",
    "biotrove-clip_common_names",
    "biotrove-clip_animalornot",
]

species_classes = [
    "Rattus norvegicus",
    "Rattus rattus",
    "Mus musculus",
    "Myodes glareolus",
    "Apodemus agrarius",
    "Apodemus flavicollis",
    "Apodemus sylvaticus",
    "Microtus arvalis",
    "Microtus agrestis",
    "Crocidura leucodon",
]

common_name_classes = [
    "brown rat",
    "black rat",
    "house mouse",
    "bank vole",
    "striped field mouse",
    "yellow-necked mouse",
    "wood mouse",
    "common vole",
    "field vole",
    "bicolored white-toothed shrew",
]

animal_or_not = [
    "This is a photo of an animal like a mouse, rat, cat, fox, or hamster",
    "This is not a photo an animal at all, but a photo of something else entirely like trash, a car, a plastic box or a piece of wood or a leaf",
]

rodent_or_not = [
    "This is a photo of a rodent like a rat, mouse or vole, or similar small mammal like a shrew",
    "This is a photo of not a rodent at all, but some other animal like a snake, bird or human ",
]

# when it has found these it has found the animal in the picture
yolo_animal_found = ["animal", "bear", "cat", "sheep", "cow", "dog", "bird"]

# when it reports these it has found the animal in the picture
species_net_animal_found = ["animal", "species", "family", "mammal"]

model_classes = {
    "yoloe_rodentornot": rodent_or_not,
    "yoloe_animalornot": animal_or_not,
    "yolo26": yolo_animal_found,
    "speciesnet": species_net_animal_found,
    "biotrove-clip_species_names": species_classes,
    "biotrove-clip_rodent_or_not": rodent_or_not,
    "biotrove-clip_common_names": common_name_classes,
    "biotrove-clip_animalornot": animal_or_not,
}

In [ ]:
data = dict()
for dp in detect_paths:
    if dp not in data:
        data[dp] = dict()

    for mp in model_paths:
        if mp not in data[dp]:
            data[dp][mp] = dict()
        path = Path(basepath) / dp / mp / "central-europe"

        for sp in path.iterdir():
            if sp.name not in data[dp][mp]:
                data[dp][mp][sp.name] = dict()
            results = json.loads((sp / "detections.json").read_text())

            for species, detects in results.items():
                for entry in detects:
                    cls = entry["class"]
                    conf = entry["conf"]

                    if mp == "yolo26":
                        if cls in [
                            "bird",
                            "bear",
                            "cat",
                            "dog",
                            "sheep",
                            "elephant",
                            "cow",
                            "giraffe",
                            "horse",
                            "zebra",
                        ]:
                            cls = "animal"
                        else:
                            cls = "nonanimal"

                    if cls not in data[dp][mp][sp.name]:
                        data[dp][mp][sp.name][cls] = dict()
                        data[dp][mp][sp.name][cls]["count"] = 1
                        data[dp][mp][sp.name][cls]["conf"] = [
                            conf,
                        ]
                    else:
                        data[dp][mp][sp.name][cls]["count"] += 1
                        data[dp][mp][sp.name][cls]["conf"].append(conf)

data

In [ ]:
import numpy as np
import pandas as pd

conf_summary_rows = []

for detect_path, models in data.items():
    for model_name, species_results in models.items():
        for species_name, class_results in species_results.items():
            for class_name, class_counts in class_results.items():
                if "conf" not in class_counts:
                    continue

                conf = np.asarray(class_counts["conf"], dtype=float)
                conf_summary_rows.append({
                    "detect_path": detect_path,
                    "model": model_name,
                    "species": species_name,
                    "class": class_name,
                    "count": conf.size,
                    "min": np.min(conf),
                    "q1": np.percentile(conf, 25),
                    "median": np.percentile(conf, 50),
                    "q3": np.percentile(conf, 75),
                    "max": np.max(conf),
                    "mean": np.mean(conf),
                })

conf_summary_df = pd.DataFrame(conf_summary_rows)
conf_summary_df

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
for key, df in conf_summary_df.groupby(["detect_path", "model", "class"]):

    value_cols = ["min", "q1", "median", "mean", "q3", "max"]

    plot_df = df.melt(
        id_vars=["species"],
        value_vars=value_cols,
        var_name="metric",
        value_name="confidence",
    )

    grid = sns.relplot(
        data=plot_df,
        x="species",
        y="confidence",
        hue="metric",
        kind="scatter",
        height=3,
        aspect=1.5,
    )

    title = key[1]

    grid.figure.suptitle(title, fontsize=16)
    grid.figure.subplots_adjust(top=0.9);
    grid.tick_params(axis="x", rotation=90)
    grid.figure.savefig(Path(basepath) / f"{key[0]}_{key[1]}_{key[2]}.jpg")
    del grid
